In [1]:
import soundfile as sf
wav, sr = sf.read("../data/audio/dev/dia113_utt10.flac", dtype="float32")
print(wav.shape, sr)   # (44000,) 16000  — 1-D, mono

(44032,) 16000


In [3]:
wav, sr

(array([-0.00041211, -0.00048792, -0.00051725, ..., -0.00152349,
        -0.00231457, -0.00262988], shape=(44032,), dtype=float32),
 16000)

In [7]:
from transformers import AutoFeatureExtractor, WavLMModel
import torch

dev = "mps" if torch.backends.mps.is_available() else "cpu" 

fe  = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").eval().to(dev)

inp = fe(wav, sampling_rate=16000, return_tensors="pt").to(dev)
with torch.no_grad():
    out = wlm(**inp, output_hidden_states=True)

len(out.hidden_states)          # 13 (embeddings + 12 layers)
out.hidden_states[0].shape      # [1, frames, 768], frames ≈ duration × 50

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


torch.Size([1, 137, 768])

In [8]:
feats = []
for h in out.hidden_states:                    # each [1, frames, 768]
    feats.append(h.mean(1))                    # [1, 768]
    feats.append(h.std(1))                     # [1, 768]
x_audio = torch.cat(feats, dim=1)              # [1, 19968]

In [11]:
x_audio.size()

torch.Size([1, 19968])

In [14]:
import numpy as np

X = np.load("../cache/audio_wavlm_13L.npy")
print(X.shape, X.dtype)
print(np.isnan(X).any(), np.isinf(X).any())
print((np.abs(X).sum(1) == 0).sum())      # rows that are all zero — should be 0
print(X[6500:6600].std())                  # around the error point — should be nonzero

(13706, 19968) float16
False False
0
inf


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:170: RuntimeWarning: overflow encountered in reduce
  arrmean = umr_sum(arr, axis, dtype, keepdims=True, where=where)
